In [16]:
import array
import pandas
import math  
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import tree
from pandas import Series, DataFrame
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn import linear_model
from sklearn import preprocessing
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from scipy import stats
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
#DEAP gene agrithm
from deap import base, creator, tools
from scipy.stats import bernoulli
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from itertools import combinations
from scipy.special import comb, perm
import json

rs = ShuffleSplit(n_splits=100, test_size=.2, random_state=0)
scaler_M = MinMaxScaler()

In [17]:
data=pandas.read_csv(r"C:\Users\z\Desktop\模型统计性质分析\二元合金体积效应数据分类分析\Outliers\Select by Pearson cofficients from initial dataset.csv")

F=np.array(data)
p=data.ev_ratio
p=np.array(p)
F_0=F[:,3:67]
sf=np.array(data.sf)/100
Vsv=np.array(data.sv_Volume)
Vso=np.array(data.so_Volume)
N_sv=np.array(data.N_sv)
N_so=np.array(data.N_so)

In [18]:
import random
random.seed(42) # 确保结果可以复现
# 2.3.5 Seeding a population
import json
from deap import base
from deap import creator
toolbox = base.Toolbox() #实例化一个Toolbox
# 描述问题
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

D:\Python\lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'FitnessMin' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
D:\Python\lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [19]:
def transe(s):
    b=[i for i,x in enumerate(s) if x==1]
    Ftr=[]
    for k in range(len(b)):
        Ftr.append(F_0[:,b[k]])
    Ftr=np.array(Ftr)
    Ftr=np.squeeze(Ftr)
    Ftr=Ftr.transpose()    
    return Ftr#加“，”变成turbl

In [20]:
def evaluate(individual):
    r=[]
    n=-1
    if individual.shape[0]==0:
            r_2=-100 
            r.append(r_2)
            #RMSE=100
            r.append(RMSE)  
    else:
        for train_index, test_index in rs.split(individual):
            if np.size(individual.shape)==1:
                scaler_M.fit(individual.reshape(-1,1)[train_index,:])
                F_tr=scaler_M.transform(individual.reshape(-1,1)[train_index,:])
                F_te=scaler_M.transform(individual.reshape(-1,1)[test_index,:])
                
                n=n+1
                krr=KernelRidge(alpha=_alpha[n], 
                                kernel='laplacian',
                                #kernel='rbf',
                                gamma=_gamma[n])
                
                krr.fit(F_tr,(p.reshape(-1,1)[train_index,:]-1)*100)
                p_pre = krr.predict(F_te)
                
                r_2=r2_score(p.reshape(-1,1)[test_index,:], p_pre) 
                #RMSE=math.pow(mean_squared_error((p.reshape(-1,1)[test_index,:]-1)*100,p_pre), 0.5)
                r.append(r_2)
                #r.append(RMSE)
            else:
                scaler_M.fit(individual[train_index,:])
                F_tr=scaler_M.transform(individual[train_index,:])
                F_te=scaler_M.transform(individual[test_index,:])
                
                n=n+1
                krr=KernelRidge(alpha=_alpha[n], 
                                kernel='laplacian',
                                #kernel='rbf',
                                gamma=_gamma[n])
                
                krr.fit(F_tr,(p.reshape(-1,1)[train_index,:]-1)*100)
                p_pre = krr.predict(F_te)
                r_2=r2_score(p.reshape(-1,1)[test_index,:], p_pre) 
                #RMSE=math.pow(mean_squared_error((p.reshape(-1,1)[test_index,:]-1)*100,p_pre), 0.5)
                r.append(r_2)
                #r.append(RMSE)
    return np.mean(r), 

In [21]:
# 注册进化过程需要的工具：配种选择、交叉、突变
toolbox.register('Evaluate', evaluate)
toolbox.register('tourSel', tools.selRoulette) 
toolbox.register('crossover', tools.cxUniform)
toolbox.register('mutate', tools.mutFlipBit)

# 用数据记录算法迭代过程
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)
logbook = tools.Logbook()

# 进化迭代
N_GEN = 50 # 迭代代数
CXPB = 0.8 # 交叉概率
MUTPB = 0.1 # 突变概率

In [22]:
a=pandas.read_csv(r"C:\Users\z\Desktop\图表及相应源代码\Fig 5\SBS AND SFS\KRR\hyper parameter container\lap\_alpha_SBS from initial-MA-clipped-17.csv")
b=pandas.read_csv(r"C:\Users\z\Desktop\图表及相应源代码\Fig 5\SBS AND SFS\KRR\hyper parameter container\lap\_gamma_SBS from initial-MA-clipped-17.csv")

_alpha=np.array(a).flatten()
_gamma=np.array(b).flatten()

In [24]:
# 接续族群
import json
from deap import base
from deap import creator

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

def initIndividual(icls, content):
    return icls(content)

def initPopulation(pcls, ind_init, filename):
    with open(filename, "r") as pop_file:
        contents = json.load(pop_file)
    return pcls(ind_init(c) for c in contents)

toolbox.register("individual_guess", initIndividual, creator.Individual)
toolbox.register("population_guess", initPopulation, list, toolbox.individual_guess, "49.json")

pop = toolbox.population_guess()

D:\Python\lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'FitnessMin' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
D:\Python\lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [25]:
evaluate_r_2=[]
evaluate_r_2_tr=[]
evaluate_RMSE=[]

m=0
n=-1

for train_index, test_index in rs.split(transe(pop[m])):
    
    scaler_M.fit(transe(pop[m])[train_index,:])
    
    F_tr=scaler_M.transform(transe(pop[m])[train_index,:])
    F_te=scaler_M.transform(transe(pop[m])[test_index,:])
        
    n=n+1
    
    krr=KernelRidge(alpha=_alpha[n], 
                    #kernel='rbf',
                    kernel='laplacian',
                    gamma=_gamma[n])
    
    krr.fit(F_tr,(p.reshape(-1,1)[train_index,:]-1)*100)
    
    p_tr = krr.predict(F_tr)
    r_2_tr=r2_score((p.reshape(-1,1)[train_index,:]-1)*100, p_tr)
    evaluate_r_2_tr.append(r_2_tr)
    
    p_pre = krr.predict(F_te)   
    r_2=r2_score((p.reshape(-1,1)[test_index,:]-1)*100, p_pre)
    evaluate_r_2.append(r_2)

print(np.mean(evaluate_r_2_tr))
print(np.std(evaluate_r_2_tr))
print(np.mean(evaluate_r_2))
print(np.std(evaluate_r_2))

0.9996243131340274
0.0015687673186122405
0.5781384465127701
0.09658388008812767


In [27]:
evaluate_r_2=[]
evaluate_RMSE=[]

#sf
evaluatesf_r_2=[]
evaluatesf_RMSE=[]

In [28]:
n=-1

for train_index, test_index in rs.split(transe(pop[m])):
    
    scaler_M.fit(transe(pop[m])[train_index,:])
    
    F_tr=scaler_M.transform(transe(pop[m])[train_index,:])
    F_te=scaler_M.transform(transe(pop[m])[test_index,:])
      
    n=n+1
    krr=KernelRidge(alpha=_alpha[n], 
                    #kernel='rbf',
                    kernel='laplacian',
                    gamma=_gamma[n])
    
    krr.fit(F_tr,(p.reshape(-1,1)[train_index,:]-1)*100)
    p_pre = krr.predict(F_te)
    
    #VLF
    r_2=r2_score((p.reshape(-1,1)[test_index,:]-1)*100, p_pre) 
    evaluate_r_2.append(r_2)
    
    #SF
    sf_pre=(((p_pre/100)+1)*Vso.reshape(-1,1)[test_index,:]-Vsv.reshape(-1,1)[test_index,:])/Vsv.reshape(-1,1)[test_index,:]
    r_2=r2_score(sf.reshape(-1,1)[test_index,:], sf_pre)
    evaluatesf_r_2.append(r_2)

In [29]:
#VLF
m_r_2=np.mean(evaluate_r_2)
s_r_2=np.std(evaluate_r_2)
print("VLF:")
print("r_2_mean=%.5f"%(m_r_2))
print("r_2_std=%.5f"%(s_r_2))

#sf
m_r_2=np.mean(evaluatesf_r_2)
s_r_2=np.std(evaluatesf_r_2)
print("SF:")
print("r_2_mean=%.5f"%(m_r_2))
print("r_2_std=%.5f"%(s_r_2))

VLF:
r_2_mean=0.57814
r_2_std=0.09658
SF:
r_2_mean=0.72220
r_2_std=0.12588


In [72]:
# 生成个体
GENE_LENGTH = 64
toolbox.register('Binary',bernoulli.rvs, 0.2)#设置初始群特征数
toolbox.register('Individual', tools.initRepeat, creator.Individual, toolbox.Binary, n=GENE_LENGTH)

# 生成初始族群
N_POP = 100
toolbox.register('Population', tools.initRepeat, list, toolbox.Individual)
pop=toolbox.Population(n = N_POP)

pop2 = json.dumps(pop, ensure_ascii=False)
f = open('./%s'%10000+'.json','w')
f.write(pop2)
f.close()

In [10]:
# 评价初始族群
fitnesses = map(toolbox.Evaluate, map(transe, pop))
for ind, fit in zip(pop, fitnesses):
    ind.fitness.values = fit

In [11]:
pop_1=pop
f = [ind.fitness.values[0] for ind in pop_1]
f_min=[ind.fitness.values[0]==min(f) for ind in pop_1]
f_min1 =f_min
fmin_index=[]

for k in range(f_min1.count(True)):
    i = f_min1.index(True)
    f_min1[i]=False
    fmin_index.append(i)

#输出选择的特征
popmin_index=[]

for k in range(pop_1[fmin_index[0]].count(1)):
    i = pop_1[fmin_index[0]].index(1)
    pop_1[fmin_index[0]][i]=0
    popmin_index.append(i)
    
popmin_index=[i+2 for i in popmin_index]

In [12]:
R=pd.DataFrame(popmin_index)
R.to_csv('popmin_index.csv')

In [ ]:
# 配种选择
for gen in range(N_GEN):
    pop_c=[]
    for t in range(100):
        selectedTour = toolbox.tourSel(pop,2) # 选择N个体
        Sel_fits=[ind.fitness.values[0] for ind in selectedTour]
        if (Sel_fits[0]-Sel_fits[1])>0:
            pop_c.append(selectedTour[1])
        else:
            pop_c.append(selectedTour[0])
        
    selectedInd = list(map(toolbox.clone, pop_c)) # 复制个体，供交叉变异用
    
# 对选出的育种族群两两进行交叉，对于被改变个体，删除其适应度值
    for child1, child2 in zip(selectedInd[::2], selectedInd[1::2]):
        if random.random() < CXPB:
            toolbox.crossover(child1, child2, 0.5)
            del child1.fitness.values
            del child2.fitness.values
            
 # 对选出的育种族群进行变异，对于被改变个体，删除适应度值
    for mutant in selectedInd:
        if random.random() < MUTPB:
            toolbox.mutate(mutant, 0.5)
            del mutant.fitness.values
        
    invalid_ind = [ind for ind in selectedInd if not ind.fitness.valid]
    
    fitnesses = map(toolbox.Evaluate, map(transe, invalid_ind))
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit
        
    pop[:] = selectedInd
    pop = json.dumps(pop, ensure_ascii=False)
    f = open('./%s'%gen + '.json','w')
    f.write(pop)
    f.close()
    
    # 完全重插入
    pop = json.loads(pop)
    pop[:] = selectedInd
    
    # 记录数据
    record = stats.compile(pop)
    logbook.record(gen=gen, **record)

In [71]:
# 输出计算过程
logbook.header = 'gen',"avg", "std", 'min', "max"
print(logbook)

gen	avg    	std     	min    	max    
0  	11.1563	0.722803	9.98032	14.4575
1  	10.8242	0.584315	9.9291 	12.8585
2  	10.5215	0.440314	9.74568	12.4984
3  	10.2888	0.362113	9.37754	11.4453
4  	10.1983	0.340075	9.49638	11.0729
5  	10.1386	0.351265	9.57832	11.2302
6  	10.0167	0.311846	9.48801	11.1087
7  	9.96605	0.332582	9.49893	10.9639
8  	9.86373	0.272012	9.28024	10.8255
9  	9.86054	0.346143	9.35282	11.1484
10 	9.81006	0.368798	9.24158	10.9711
11 	9.83   	0.398361	9.24565	11.1113
12 	9.66988	0.323182	9.21059	11.0381
13 	9.62479	0.359912	9.01335	10.8837
14 	9.62028	0.381873	9.01335	10.6938
15 	9.58444	0.416611	9.01335	11.3816
16 	9.51866	0.40723 	9.01335	11.3003
17 	9.47426	0.344259	9.0693 	10.7216
18 	9.42536	0.394063	9.00057	10.8788
19 	9.38875	0.451198	9.00287	10.9565
20 	9.34223	0.430576	9.02154	11.5476
21 	9.23178	0.270144	8.93264	10.5745
22 	9.26482	0.438646	8.92848	11.0072
23 	9.31585	0.504983	8.90574	10.9174
24 	9.30715	0.523735	8.89789	10.9505
25 	9.30262	0.562929	8.89789	11.4808
2

In [159]:
r=pd.DataFrame(transe(pop[0]))
r.to_csv('GA_selected.csv')